Workflow plan:

- Takes an area geojson, like example_datasets/Leeds_pp_or_g_cmb.geojson
- Checks if it's Wales or Leeds using the LUT desaigned in data_structuring repository (example_datasets/all_parks_ids.csv)
- Uses appropriate workflow (Wales vs England)
- Outputs all data in a folder withy the same prefix as the area geojson
- All datasets have the unique park id from the LUT as a prefix to their name


In [1]:
import geopandas as gpd
import json
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt


sys.path.insert(0, '../src')

import park_vga

In [2]:
boundaries_folder = "../example_datasets/"
boundaries_filename = "Blaenau Gwent_pp_or_g_cmb.geojson"
boundaries_file = Path(boundaries_folder) / boundaries_filename

eng_dtm = "/Volumes/Extreme SSD/DTM"
eng_dsm = "/Volumes/Extreme SSD/FZ_DSM"
wales_dtm = "/Volumes/Extreme SSD/wales_lidar/wales_dtm_32bit_cog.tif"
wales_dsm = "/Volumes/Extreme SSD/wales_lidar/wales_dsm_32bit_cog.tif"

output_folder = "../workflow_outputs/"

check_regions_file = "../example_datasets/LUT_regions_authorities_filenames.geojson"
park_ids_file = "../example_datasets/all_parks_ids.csv"

In [3]:
# load the boundaries file as a geodataframe
gdf = gpd.read_file(boundaries_file)

# pull out the title of the boundaries file (after last "/" and before ".geojson")
boundaries_title = boundaries_filename.split(".")[0]
print(boundaries_title)

output_path = Path(output_folder) / boundaries_title
print(output_path)
#check that output path exists, and if not, create it
if not output_path.exists():
    output_path.mkdir(parents=True, exist_ok=True)

# use check_regions_file  to check country for the boundaries file
# see if [filename] matches boundaries_filename in check_regions_file, and if so, pull out the country
check_regions_gdf = gpd.read_file(check_regions_file)
check_regions_gdf["filename"] = check_regions_gdf["filename"].apply(lambda x: Path(x).name)
country = check_regions_gdf.loc[check_regions_gdf["filename"] == boundaries_filename, "country"].values[0]
authority = check_regions_gdf.loc[check_regions_gdf["filename"] == boundaries_filename, "auth_name_e"].values[0]
print(country, authority)


# filter the park ids dataframe to just the authority
park_ids_file_df = pd.read_csv(park_ids_file)
# subset to where auth_name_e matches authority
park_ids_file_df = park_ids_file_df.loc[park_ids_file_df["auth_name_e"] == authority]
print(len(park_ids_file_df))


Blaenau Gwent_pp_or_g_cmb
../workflow_outputs/Blaenau Gwent_pp_or_g_cmb
Wales Blaenau Gwent
55


In [4]:
park_ids_file_df

,country,region_id,authority_id,auth_name_e,old_park_id,new_park_id
4448,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_82e0ba259db8;BLAGW_704b870f597a,9d36dfd30e01
4449,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_2d73de2ce881,3cff40fe6053
4450,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_c73b3efa4f2b;BLAGW_a09e8e71ee9e;BLAGW_ca...,91e51f3c4786
4451,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_175c584682e9,5d5a1d1eddd5
4452,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_bc0a2bc57236;BLAGW_fda80533c869,aa4af773404c
4453,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_333c05376af0,73706a1dbc90
4454,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_d124f4b62479,29b9666a6a9e
4455,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_023f9ffc2443,367dbfce9a2a
4456,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_2efda72621e0;BLAGW_494a3f406b18,0a5c6a66d4e4
4457,Wales,W10000008,W06000019,Blaenau Gwent,BLAGW_96f59354df18,58f0b530daa9


In [ ]:
for n in range(len(gdf)):
    print(f"{n}/{len(gdf)}")
    park_id = gdf.loc[n, "id"]
    print(park_id)
    # use subsetted park ids dataframe to find the row where park_id matches id in park_ids_file_df
    # and pull out new park id (safe for filenames)
    park_id_safe = park_ids_file_df.loc[park_ids_file_df["old_park_id"] == park_id, "new_park_id"].values[0]
    print(park_id_safe)
    # check if output files already exist for this park, and if so, skip to the next park
    output_file = output_path / f"{park_id_safe}_visibility.geojson"
    if output_file.exists():
        print(f"Output file {output_file} already exists, skipping park {park_id_safe}")
        continue

    if country == "England":
        dtm_path = eng_dtm
        dsm_path = eng_dsm
        results = park_vga.workflow.workflow_eng(boundaries_file, n,
                                            dtm_path, dsm_path,
                                            output_path,
                                            spacing=12,
                                            return_results=False, save_results=True,
                                            park_id_for_file_name=park_id_safe,
                                            max_distance=80)


    elif country == "Wales":
        dtm_path = wales_dtm
        dsm_path = wales_dsm
        results = park_vga.workflow.workflow_wales(boundaries_file, n,
                                            dtm_path, dsm_path,
                                            output_path,
                                            spacing=12,
                                            return_results=False, save_results=True,
                                            park_id_for_file_name=park_id_safe,
                                            max_distance=80)


0/55
BLAGW_82e0ba259db8;BLAGW_704b870f597a
9d36dfd30e01
Output directory already exists at ../workflow_outputs/Blaenau Gwent_pp_or_g_cmb
Park ID: BLAGW_82e0ba259db8;BLAGW_704b870f597a
Park ID for file name: 9d36dfd30e01
Define park grid.
Load park tiles
Created 133 pointy-top hexagons (12m vertical spacing)
Extracted (219, 119) pixels, Height range: -0.1m to 20.8m
Start calculating visibility. This will take a few mins
Calculating visibility for 133 points
  Pre-computing elevations for all points...
  Building spatial index...
  Processing point 1/133
  Processing point 51/133
  Processing point 101/133
Visibility results saved to ../workflow_outputs/Blaenau Gwent_pp_or_g_cmb/9d36dfd30e01_visibility.geojson
Calculating foliage statistics
Extracting foliage data from 133 hexagons...
  Processed 13 / 133 hexagons
  Processed 26 / 133 hexagons
  Processed 39 / 133 hexagons
  Processed 52 / 133 hexagons
  Processed 65 / 133 hexagons
  Processed 78 / 133 hexagons
  Processed 91 / 133 hexag